# Facial Verification with a Siamese Network

Reconstructed, cleaned-up version of the original project notebook. This implements one-shot face verification using a Siamese neural network trained on anchor/positive/negative image triplets, following the approach of Koch et al. (Siamese Neural Networks for One-shot Image Recognition).

**Pipeline:** collect images -> preprocess -> build embedding + Siamese model -> train -> evaluate -> save -> real-time webcam verification.

## 1. Setup
### 1.1 Install Dependencies
Run this once. See `requirements.txt` for pinned versions.

In [ ]:
# !pip install -r requirements.txt

### 1.2 Import Dependencies

In [ ]:
# Import standard dependencies
import os
import uuid
import random

import cv2
import numpy as np
from matplotlib import pyplot as plt

# Import tensorflow dependencies - Functional API
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, Dense, MaxPooling2D, Input, Flatten
from tensorflow.keras.metrics import Precision, Recall

# Custom L1 distance layer (kept in its own module - see layers.py -
# this is required so the saved model can be reloaded later)
from layers import L1Dist


### 1.3 Set GPU Growth
Prevents TensorFlow from grabbing all GPU memory at once (avoids OOM errors if you have other processes using the GPU).

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f"GPUs available: {len(gpus)}")


### 1.4 Create Folder Structure
`exist_ok=True` fixes the `FileExistsError` from the original notebook if you re-run this cell.

In [ ]:
POS_PATH = os.path.join('data', 'positive')
NEG_PATH = os.path.join('data', 'negative')
ANC_PATH = os.path.join('data', 'anchor')

os.makedirs(POS_PATH, exist_ok=True)
os.makedirs(NEG_PATH, exist_ok=True)
os.makedirs(ANC_PATH, exist_ok=True)


## 2. Collect Positives and Anchors
### 2.1 Untar Labelled Faces in the Wild (LFW) Dataset
Download from http://vis-www.cs.umass.edu/lfw/ (the `lfw.tgz` file) and place it in the project root before running this cell. LFW images are used as the **negative** class (i.e. "not the same person").

In [ ]:
import tarfile

if os.path.exists('lfw.tgz') and not os.path.exists('lfw'):
    with tarfile.open('lfw.tgz') as tar:
        tar.extractall()


In [ ]:
# Move LFW images into data/negative
if os.path.exists('lfw'):
    for directory in os.listdir('lfw'):
        dir_path = os.path.join('lfw', directory)
        if not os.path.isdir(dir_path):
            continue
        for file in os.listdir(dir_path):
            ex_path = os.path.join(dir_path, file)
            new_path = os.path.join(NEG_PATH, file)
            os.replace(ex_path, new_path)


### 2.2 Collect Positive and Anchor Classes via Webcam
Press **`a`** to capture an anchor image, **`p`** to capture a positive (same-person) image, and **`q`** to quit.

> This needs a webcam attached to the machine running the notebook, so it won't work in a headless/cloud environment - run it locally.

In [ ]:
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Cut down frame to 250x250px
    frame = frame[120:120+250, 200:200+250, :]

    # Collect anchors
    if cv2.waitKey(1) & 0xFF == ord('a'):
        imgname = os.path.join(ANC_PATH, f'{uuid.uuid1()}.jpg')
        cv2.imwrite(imgname, frame)

    # Collect positives
    if cv2.waitKey(1) & 0xFF == ord('p'):
        imgname = os.path.join(POS_PATH, f'{uuid.uuid1()}.jpg')
        cv2.imwrite(imgname, frame)

    cv2.imshow('Image Collection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


## 3. Load and Preprocess Images
### 3.1 Get Image Directories

In [ ]:
anchor = tf.data.Dataset.list_files(ANC_PATH + '/*.jpg').take(300)
positive = tf.data.Dataset.list_files(POS_PATH + '/*.jpg').take(300)
negative = tf.data.Dataset.list_files(NEG_PATH + '/*.jpg').take(300)


### 3.2 Preprocessing - Scale and Resize

In [ ]:
def preprocess(file_path):
    # Read in image from file path
    byte_img = tf.io.read_file(file_path)
    # Load in the image
    img = tf.io.decode_jpeg(byte_img)

    # Resize to 100x100x3
    img = tf.image.resize(img, (100, 100))
    # Scale image to be between 0 and 1
    img = img / 255.0

    return img


### 3.3 Create Labelled Dataset
(anchor, positive) pairs are labelled `1`, (anchor, negative) pairs are labelled `0`.

In [ ]:
positives = tf.data.Dataset.zip((
    anchor, positive, tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))
))
negatives = tf.data.Dataset.zip((
    anchor, negative, tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor)))
))
data = positives.concatenate(negatives)


### 3.4 Build Train and Test Partition

In [ ]:
def preprocess_twin(input_img, validation_img, label):
    return (preprocess(input_img), preprocess(validation_img), label)


# Build dataloader pipeline
data = data.map(preprocess_twin)
data = data.cache()
data = data.shuffle(buffer_size=1024)

# Training partition
train_data = data.take(round(len(data) * .7))
train_data = train_data.batch(16)
train_data = train_data.prefetch(8)

# Testing partition
test_data = data.skip(round(len(data) * .7))
test_data = test_data.take(round(len(data) * .3))
test_data = test_data.batch(16)
test_data = test_data.prefetch(8)


## 4. Model Engineering
### 4.1 Build Embedding Layer
A 4-block CNN that turns a 100x100x3 face image into a 4096-dimensional embedding vector.

In [ ]:
def make_embedding():
    inp = Input(shape=(100, 100, 3), name='input_image')

    # First block
    c1 = Conv2D(64, (10, 10), activation='relu')(inp)
    m1 = MaxPooling2D(64, (2, 2), padding='same')(c1)

    # Second block
    c2 = Conv2D(128, (7, 7), activation='relu')(m1)
    m2 = MaxPooling2D(64, (2, 2), padding='same')(c2)

    # Third block
    c3 = Conv2D(128, (4, 4), activation='relu')(m2)
    m3 = MaxPooling2D(64, (2, 2), padding='same')(c3)

    # Final embedding block
    c4 = Conv2D(256, (4, 4), activation='relu')(m3)
    f1 = Flatten()(c4)
    d1 = Dense(4096, activation='sigmoid')(f1)

    return Model(inputs=[inp], outputs=[d1], name='embedding')


embedding = make_embedding()
embedding.summary()


### 4.2 Build Distance Layer
`L1Dist` is imported from `layers.py` (see Section 1.2) rather than defined inline, so the trained model can be reloaded cleanly later.

### 4.3 Make Siamese Model

In [ ]:
def make_siamese_model():
    # Anchor image input in the network
    input_image = Input(name='input_img', shape=(100, 100, 3))

    # Validation image in the network
    validation_image = Input(name='validation_img', shape=(100, 100, 3))

    # Combine siamese distance components
    siamese_layer = L1Dist()
    siamese_layer._name = 'distance'
    distances = siamese_layer([embedding(input_image), embedding(validation_image)])

    # Classification layer
    classifier = Dense(1, activation='sigmoid')(distances)

    return Model(
        inputs=[input_image, validation_image],
        outputs=classifier,
        name='SiameseNetwork'
    )


siamese_model = make_siamese_model()
siamese_model.summary()


## 5. Training
### 5.1 Setup Loss and Optimizer

In [ ]:
binary_cross_loss = tf.losses.BinaryCrossentropy()
opt = tf.keras.optimizers.Adam(1e-4)  # 0.0001


### 5.2 Establish Checkpoints

In [ ]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, 'ckpt')
checkpoint = tf.train.Checkpoint(opt=opt, siamese_model=siamese_model)


### 5.3 Build Train Step Function

In [ ]:
@tf.function
def train_step(batch):
    # Record all of our operations
    with tf.GradientTape() as tape:
        # Get anchor and positive/negative image
        X = batch[:2]
        # Get label
        y = batch[2]

        # Forward pass
        yhat = siamese_model(X, training=True)
        # Calculate loss
        loss = binary_cross_loss(y, yhat)

    # Calculate gradients
    grad = tape.gradient(loss, siamese_model.trainable_variables)

    # Calculate updated weights and apply to siamese model
    opt.apply_gradients(zip(grad, siamese_model.trainable_variables))

    return loss


### 5.4 Build Training Loop

In [ ]:
def train(data, EPOCHS):
    for epoch in range(1, EPOCHS + 1):
        print(f'\n Epoch {epoch}/{EPOCHS}')
        progbar = tf.keras.utils.Progbar(len(data))

        for idx, batch in enumerate(data):
            train_step(batch)
            progbar.update(idx + 1)

        # Save checkpoints every 10 epochs
        if epoch % 10 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)


### 5.5 Train the Model

In [ ]:
EPOCHS = 50
train(train_data, EPOCHS)


## 6. Evaluate Model
### 6.1 Make Predictions

In [ ]:
# Get a batch of test data
test_input, test_val, y_true = test_data.as_numpy_iterator().next()

# Make predictions
y_hat = siamese_model.predict([test_input, test_val])

# Post-process results (threshold at 0.5)
predictions = [1 if prediction > 0.5 else 0 for prediction in y_hat]
print("Predictions:", predictions)
print("Ground truth:", y_true)


### 6.2 Calculate Metrics

In [ ]:
m_recall = Recall()
m_recall.update_state(y_true, y_hat)
print("Recall:", m_recall.result().numpy())

m_precision = Precision()
m_precision.update_state(y_true, y_hat)
print("Precision:", m_precision.result().numpy())


### 6.3 Visualize Results

In [ ]:
plt.figure(figsize=(10, 8))

plt.subplot(1, 2, 1)
plt.imshow(test_input[0])
plt.title("Input")

plt.subplot(1, 2, 2)
plt.imshow(test_val[0])
plt.title("Validation")

plt.show()


## 7. Save Model

In [ ]:
siamese_model.save('siamesemodel.h5')


In [ ]:
# Reload model to confirm it works end-to-end
model = tf.keras.models.load_model(
    'siamesemodel.h5',
    custom_objects={'L1Dist': L1Dist, 'BinaryCrossentropy': tf.losses.BinaryCrossentropy}
)
model.summary()


## 8. Real-Time Verification
### 8.1 Verification Function
Compares a freshly captured `input_image` against every image in `application_data/verification_images/` (photos of the person you want to be able to verify). `detection_threshold` is the per-image similarity score cutoff; `verification_threshold` is the proportion of verification images that must pass for an overall "verified" result.

In [ ]:
def verify(model, detection_threshold, verification_threshold):
    results = []
    verification_dir = os.path.join('application_data', 'verification_images')
    input_path = os.path.join('application_data', 'input_image', 'input_image.jpg')

    for image in os.listdir(verification_dir):
        input_img = preprocess(input_path)
        validation_img = preprocess(os.path.join(verification_dir, image))

        result = model.predict(
            list(np.expand_dims([input_img, validation_img], axis=1))
        )
        results.append(result)

    detection = np.sum(np.array(results) > detection_threshold)
    verification = detection / len(os.listdir(verification_dir))
    verified = verification > verification_threshold

    return results, verified


### 8.2 OpenCV Real-Time Verification
Press **`v`** to capture the current frame and run verification against your reference images, **`q`** to quit.

In [ ]:
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = frame[120:120+250, 200:200+250, :]
    cv2.imshow('Verification', frame)

    # Verification trigger
    if cv2.waitKey(10) & 0xFF == ord('v'):
        input_path = os.path.join('application_data', 'input_image', 'input_image.jpg')
        cv2.imwrite(input_path, frame)

        results, verified = verify(model, 0.9, 0.7)
        print("Verified:", verified)

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
